# Exploring `mujoco/invertedpendulum/expert-v0` with Minari

This notebook loads the Minari expert dataset for the MuJoCo InvertedPendulum environment,
assembles a **100 000 × 5** observation-action matrix, and lays the groundwork for training
a Self-Organizing Map (SOM) on the resulting data.

| Column index | Description |
|---|---|
| 0 | Cart position |
| 1 | Cart velocity |
| 2 | Pole angle |
| 3 | Pole angular velocity |
| 4 | Action (force applied to cart) |

## 1  Imports

In [1]:
import numpy as np
import minari
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 2  Load dataset

If the dataset has not been downloaded yet, Minari will fetch it automatically from the remote
registry the first time `load_dataset` is called.

In [2]:
DATASET_ID = "mujoco/invertedpendulum/expert-v0"

# Download if not already cached locally
if DATASET_ID not in minari.list_local_datasets():
    minari.download_dataset(DATASET_ID)

dataset = minari.load_dataset(DATASET_ID)
print(f"Dataset loaded: {DATASET_ID}")
print(f"Total episodes : {dataset.total_episodes}")
print(f"Total steps    : {dataset.total_steps}")

Dataset loaded: mujoco/invertedpendulum/expert-v0
Total episodes : 100
Total steps    : 100000


## 3  Inspect one episode

Each episode is an `EpisodeData` object with numpy arrays for `observations`, `actions`,
`rewards`, `terminations`, and `truncations`.

> **Note**: Minari stores one *extra* observation per episode (the terminal observation is
> appended), so `observations` has shape `(T+1, obs_dim)` while `actions` has shape
> `(T, act_dim)`.  We use only the first `T` observations to form aligned pairs.

In [3]:
sample_episode = next(iter(dataset.iterate_episodes()))

print("observations shape :", sample_episode.observations.shape)
print("actions shape      :", sample_episode.actions.shape)
print("rewards shape      :", sample_episode.rewards.shape)

observations shape : (1001, 4)
actions shape      : (1000, 1)
rewards shape      : (1000,)


## 4  Build the 100 000 × 5 observation-action matrix

In [4]:
TARGET_ROWS = 100_000

obs_list = []
act_list = []

for episode in dataset.iterate_episodes():
    obs = episode.observations  # shape (T+1, 4)
    act = episode.actions       # shape (T, 1)
    T = act.shape[0]
    obs_list.append(obs[:T])    # align: drop terminal observation
    act_list.append(act)

    collected = sum(a.shape[0] for a in act_list)
    if collected >= TARGET_ROWS:
        break

all_obs = np.concatenate(obs_list, axis=0)   # (N, 4)
all_act = np.concatenate(act_list, axis=0)   # (N, 1)

# Trim or pad to exactly TARGET_ROWS
all_obs = all_obs[:TARGET_ROWS]
all_act = all_act[:TARGET_ROWS]

# Combine into a single (100_000, 5) matrix
data_matrix = np.hstack([all_obs, all_act])  # (100_000, 5)

print("data_matrix shape:", data_matrix.shape)
assert data_matrix.shape == (TARGET_ROWS, 5), (
    f"Expected ({TARGET_ROWS}, 5) but got {data_matrix.shape}"
)

data_matrix shape: (100000, 5)


## 5  Summary statistics

In [5]:
COL_NAMES = ["cart_pos", "cart_vel", "pole_angle", "pole_ang_vel", "action"]

print(f"{'Column':<18} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print("-" * 62)
for i, name in enumerate(COL_NAMES):
    col = data_matrix[:, i]
    print(f"{name:<18} {col.min():>10.4f} {col.max():>10.4f} "
          f"{col.mean():>10.4f} {col.std():>10.4f}")

Column                    Min        Max       Mean        Std
--------------------------------------------------------------
cart_pos              -0.1436     0.0615    -0.0858     0.0188
cart_vel              -0.1133     0.0909    -0.0017     0.0175
pole_angle            -1.2344     0.8540    -0.0022     0.2325
pole_ang_vel          -1.8871     2.5151    -0.0000     0.5261
action                -2.8613     2.9583    -0.0000     1.0576


## 6  Normalise with scikit-learn `StandardScaler`

In [6]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_matrix)  # (100_000, 5), zero-mean unit-variance

print("Scaled data — mean per column :", data_scaled.mean(axis=0).round(6))
print("Scaled data — std  per column :", data_scaled.std(axis=0).round(6))

Scaled data — mean per column : [ 0. -0. -0. -0.  0.]
Scaled data — std  per column : [1. 1. 1. 1. 1.]


## 7  Quick PCA sanity check (scikit-learn)

In [7]:
pca = PCA(n_components=5)
pca.fit(data_scaled)

print("Explained variance ratio per component:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f}  (cumulative: {pca.explained_variance_ratio_[:i+1].sum():.4f})")

Explained variance ratio per component:
  PC1: 0.5435  (cumulative: 0.5435)
  PC2: 0.3367  (cumulative: 0.8802)
  PC3: 0.0780  (cumulative: 0.9581)
  PC4: 0.0408  (cumulative: 0.9989)
  PC5: 0.0011  (cumulative: 1.0000)


## 8  Train a Self-Organizing Map (MiniSom)

A small 10 × 10 SOM is trained on the scaled data as a starting-point for exploration.
Adjust `som_x`, `som_y`, `sigma`, `learning_rate`, and `num_iteration` as needed.

In [8]:
SOM_X = 10
SOM_Y = 10
INPUT_DIM = data_scaled.shape[1]  # 5

som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=INPUT_DIM,
    sigma=1.5,
    learning_rate=0.5,
    random_seed=42,
)

som.random_weights_init(data_scaled)
som.train(data_scaled, num_iteration=10_000, verbose=True)

print("\nSOM training complete.")
print(f"Weight matrix shape: {som.get_weights().shape}")

 [ 10000 / 10000 ] 100% - 0:00:00 left 
 quantization error: 0.6209132821060092

SOM training complete.
Weight matrix shape: (10, 10, 5)


## 9  Quantisation error

The quantisation error is the mean distance between each data point and its Best Matching
Unit (BMU) — a lower value indicates a better-fitting map.

In [9]:
q_error = som.quantization_error(data_scaled)
print(f"Quantisation error: {q_error:.6f}")

Quantisation error: 0.620913
